In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from scipy.optimize import minimize_scalar
from scipy.linalg import sqrtm

In [3]:
np.random.seed(42)  # For reproducibility

# ==============================================================================
# 1. Analytical Environments (Multi-SDE Setup)
# We define three distinct bounded scalar potentials A(x) to test varying 
# dynamics: multimodality, robust saturating gradients, and metastability.
# ==============================================================================

class AnalyticalSDE:
    def __init__(self, name, A_func, drift_func, ddrift_func):
        self.name = name
        self.A = A_func
        self.drift = drift_func
        self.ddrift = ddrift_func
        
        # Numerically solve for theoretical bounds to avoid manual derivations
        res_A_max = minimize_scalar(lambda x: -self.A(x), bounds=(-10, 10), method='bounded')
        self.A_max = self.A(res_A_max.x)
        
        def phi(x): return 0.5 * self.drift(x)**2 + 0.5 * self.ddrift(x)
        res_phi_min = minimize_scalar(lambda x: phi(x), bounds=(-10, 10), method='bounded')
        res_phi_max = minimize_scalar(lambda x: -phi(x), bounds=(-10, 10), method='bounded')
        
        self.phi_min = phi(res_phi_min.x)
        self.M_intensity = phi(res_phi_max.x) - self.phi_min
        
    def phi_tilde(self, x):
        """Shifted intensity to ensure strict non-negativity."""
        return 0.5 * self.drift(x)**2 + 0.5 * self.ddrift(x) - self.phi_min

# Model 1: Periodic (Multi-Modal)
sde1 = AnalyticalSDE("Periodic", 
                     lambda x: np.cos(x), 
                     lambda x: -np.sin(x), 
                     lambda x: -np.cos(x))

# Model 2: Pseudo-Huber (Bounded Score Proxy)
sde2 = AnalyticalSDE("Sigmoidal Proxy", 
                     lambda x: -2 * np.log(np.cosh(x)), 
                     lambda x: -2 * np.tanh(x), 
                     lambda x: -2 / np.cosh(x)**2)

# Model 3: Asymmetric Periodic
sde3 = AnalyticalSDE("Asymmetric Periodic", 
                     lambda x: np.sin(x) + 0.5 * np.sin(2*x), 
                     lambda x: np.cos(x) + np.cos(2*x), 
                     lambda x: -np.sin(x) - 2*np.sin(2*x))

In [4]:
# ==============================================================================
# 2. Simulation Solvers
# ==============================================================================

def euler_maruyama(sde, y0, N_samples, dt, T):
    """Standard Euler-Maruyama discretization baseline."""
    y = np.full(N_samples, float(y0))
    n_steps = int(T / dt)
    nfe = 0
    for _ in range(n_steps):
        y += sde.drift(y) * dt + np.sqrt(dt) * np.random.randn(N_samples)
        nfe += 1
    return y, nfe

def segs_exact_sampler(sde, y0, N_samples, dtau, T):
    """Sequential Exact Generative Sampling (SEGS) algorithm."""
    y = np.full(N_samples, float(y0))
    n_steps = int(np.ceil(T / dtau))
    total_nfe = 0
    
    for step in range(n_steps):
        dt_curr = min(dtau, T - step * dtau)
        active_idx = np.arange(N_samples)
        y_next = np.zeros(N_samples)
        
        while len(active_idx) > 0:
            K_active = len(active_idx)
            
            # --- PHASE 1: Propose Gaussian Endpoints ---
            y_cand = y[active_idx] + np.sqrt(dt_curr) * np.random.randn(K_active)
            total_nfe += K_active 
            
            # --- PHASE 2: Endpoint Acceptance ---
            prob_end = np.exp(sde.A(y_cand) - sde.A_max)
            end_mask = np.random.rand(K_active) <= prob_end
            
            idx_pass = active_idx[end_mask]
            cand_pass = y_cand[end_mask]
            K_pass = len(idx_pass)
            
            if K_pass == 0: continue
                
            # --- PHASE 3: Poisson Stochastic Thinning ---
            K_arrivals = np.random.poisson(sde.M_intensity * dt_curr, size=K_pass)
            path_accepted = np.ones(K_pass, dtype=bool)
            
            for j in range(K_pass):
                k_arr = K_arrivals[j]
                if k_arr == 0: continue
                    
                s_times = np.random.uniform(0, dt_curr, k_arr)
                Z = np.random.randn(k_arr)
                
                # Brownian bridge interpolation
                b_mean = y[idx_pass[j]] + (s_times / dt_curr) * (cand_pass[j] - y[idx_pass[j]])
                b_var = s_times * (dt_curr - s_times) / dt_curr
                B_s = b_mean + np.sqrt(b_var) * Z
                
                total_nfe += k_arr
                intensities = sde.phi_tilde(B_s)
                
                # Reject path if any Poisson thinning event is realized
                if np.any(np.random.rand(k_arr) <= intensities / sde.M_intensity):
                    path_accepted[j] = False
                    
            final_idx = idx_pass[path_accepted]
            y_next[final_idx] = cand_pass[path_accepted]
            active_idx = active_idx[~np.isin(active_idx, final_idx)]
            
        y = y_next
    return y, total_nfe / N_samples

In [5]:
# ==============================================================================
# 3. Execution & Metrics Collection
# ==============================================================================
N_SAMPLES, T_END, Y0 = 20000, 2.0, 0.0
results = []

for sde in [sde1, sde2, sde3]:
    # 1. Ground truth via ultra-fine EM (N=100k, dt=0.001)
    ref_y, _ = euler_maruyama(sde, Y0, 100000, 0.001, T_END)
    
    # 2. EM baseline (dt=0.1)
    em_y, em_nfe = euler_maruyama(sde, Y0, N_SAMPLES, 0.1, T_END)
    em_ks = ks_2samp(em_y, ref_y).statistic
    em_w2 = np.sqrt(np.mean((np.sort(em_y) - np.sort(ref_y[:N_SAMPLES]))**2))
    
    # 3. SEGS (dtau=0.5)
    segs_y, segs_nfe = segs_exact_sampler(sde, Y0, N_SAMPLES, 0.5, T_END)
    segs_ks = ks_2samp(segs_y, ref_y).statistic
    segs_w2 = np.sqrt(np.mean((np.sort(segs_y) - np.sort(ref_y[:N_SAMPLES]))**2))
    
    results.append({
        "Topology": sde.name, 
        "EM KS": em_ks, "SEGS KS": segs_ks,
        "EM W2": em_w2, "SEGS W2": segs_w2,
        "EM NFE": em_nfe, "SEGS NFE": segs_nfe
    })

result = pd.DataFrame(results)

In [6]:
result

,Topology,EM KS,SEGS KS,EM W2,SEGS W2,EM NFE,SEGS NFE
0,Periodic,0.01475,0.00522,0.029042,0.024507,20,8.80355
1,Sigmoidal Proxy,0.02052,0.00514,0.029111,0.009405,20,16.65425
2,Asymmetric Periodic,0.03079,0.00678,0.147690,0.047485,20,32.87655


In [29]:
# Ensure absolute reproducibility
np.random.seed(42)

# ==============================================================================
# 1. Rigorous High-Dimensional Metrics (Generalizing KS and W2)
# ==============================================================================
def compute_bures_wasserstein_distance(X, Y):
    """Computes the Bures-Wasserstein-2 Distance (Fréchet Pixel Distance)."""
    mu_X, sig_X = np.mean(X, axis=0), np.cov(X, rowvar=False)
    mu_Y, sig_Y = np.mean(Y, axis=0), np.cov(Y, rowvar=False)
    
    ssdiff = np.sum((mu_X - mu_Y)**2.0)
    covmean = sqrtm(sig_X.dot(sig_Y))
    if np.iscomplexobj(covmean): covmean = covmean.real
    
    return ssdiff + np.trace(sig_X + sig_Y - 2.0 * covmean)

def compute_mmd(X, Y, gamma=0.1):
    """Computes MMD, the high-dimensional generalization of the two-sample KS test."""
    XX = np.sum(X**2, axis=1)[:, None] + np.sum(X**2, axis=1)[None, :] - 2 * np.dot(X, X.T)
    YY = np.sum(Y**2, axis=1)[:, None] + np.sum(Y**2, axis=1)[None, :] - 2 * np.dot(Y, Y.T)
    XY = np.sum(X**2, axis=1)[:, None] + np.sum(Y**2, axis=1)[None, :] - 2 * np.dot(X, Y.T)
    
    return np.exp(-gamma * XX).mean() + np.exp(-gamma * YY).mean() - 2 * np.exp(-gamma * XY).mean()

In [30]:
# ==============================================================================
# 2. Factorizable 16-D Target Proxy (Stiff SDE)
# ==============================================================================
D_DIM = 16
C = 0.25  # Sharpened parameter creates stiff gradients

target_pattern = np.array([
    1, -1, 1, -1,
   -1, 1, -1, 1,
    1, -1, 1, -1,
   -1, 1, -1, 1
], dtype=np.float32)

def A_1d(x, tgt): return -np.sqrt((x - tgt)**2 + C**2)
def drift_1d(x, tgt): return -(x - tgt) / np.sqrt((x - tgt)**2 + C**2)
def lap_1d(x, tgt): return -C**2 / ((x - tgt)**2 + C**2)**1.5
def phi_1d(x, tgt): return 0.5 * drift_1d(x, tgt)**2 + 0.5 * lap_1d(x, tgt) + 1.0 / (2 * C)

A_MAX_1D = -C
M_INT_1D = 0.5 + 1.0 / (2 * C) 

In [31]:
# ==============================================================================
# 3. Solvers
# ==============================================================================
def euler_maruyama(y_noisy, dt, T):
    np.random.seed(42)
    y = y_noisy.copy()
    N_samples = y.shape[0]
    n_steps = int(T / dt)
    nfe = 0
    for _ in range(n_steps):
        diff = y - target_pattern
        drift = -diff / np.sqrt(diff**2 + C**2)
        y += drift * dt + np.sqrt(dt) * np.random.randn(N_samples, D_DIM)
        nfe += 1
    return y, nfe

def segs_16d_sampler(y_noisy, dtau, T):
    np.random.seed(42)
    # SPATIAL FACTORIZATION
    y_flat = y_noisy.copy().flatten()
    N_samples = y_noisy.shape[0]
    tgt_flat = np.tile(target_pattern, N_samples)
    N_total = len(y_flat)
    
    n_steps = int(np.ceil(T / dtau))
    total_scalar_nfe = 0
    
    for step in range(n_steps):
        dt_curr = min(dtau, T - step * dtau)
        active_idx = np.arange(N_total)
        y_next_flat = np.zeros(N_total)
        
        while len(active_idx) > 0:
            K_active = len(active_idx)
            y_cand = y_flat[active_idx] + np.sqrt(dt_curr) * np.random.randn(K_active)
            total_scalar_nfe += K_active
            
            prob_end = np.exp(A_1d(y_cand, tgt_flat[active_idx]) - A_MAX_1D)
            end_mask = np.random.rand(K_active) <= prob_end
            
            idx_pass = active_idx[end_mask]
            start_pass, cand_pass = y_flat[idx_pass], y_cand[end_mask]
            tgt_pass = tgt_flat[idx_pass]
            K_pass = len(idx_pass)
            
            if K_pass == 0: continue
            
            K_arrivals = np.random.poisson(M_INT_1D * dt_curr, size=K_pass)
            accepted = np.ones(K_pass, dtype=bool)
            
            for j in range(K_pass):
                k_arr = K_arrivals[j]
                if k_arr == 0: continue
                
                # Ensure sequential simulation of Brownian Bridge to preserve temporal correlation
                s_times = np.sort(np.random.uniform(0, dt_curr, k_arr))
                curr_y, curr_t = start_pass[j], 0.0
                y_end = cand_pass[j]
                reject = False
                
                for k in range(k_arr):
                    t_next = s_times[k]
                    dt_rem = dt_curr - curr_t
                    
                    mean_next = curr_y + (t_next - curr_t) / dt_rem * (y_end - curr_y)
                    var_next = (t_next - curr_t) * (dt_curr - t_next) / dt_rem
                    
                    next_y = mean_next + np.sqrt(max(var_next, 1e-12)) * np.random.randn()
                    curr_y, curr_t = next_y, t_next
                    total_scalar_nfe += 1
                    
                    intensity = phi_1d(next_y, tgt_pass[j])
                    if np.random.rand() <= intensity / M_INT_1D:
                        reject = True
                        break
                        
                if reject: accepted[j] = False
            
            final_idx = idx_pass[accepted]
            y_next_flat[final_idx] = cand_pass[accepted]
            active_idx = active_idx[~np.isin(active_idx, final_idx)]
            
        y_flat = y_next_flat
    
    vector_nfe = (total_scalar_nfe / D_DIM) / N_samples
    return y_flat.reshape(N_samples, D_DIM), vector_nfe

In [32]:
# ==============================================================================
# 4. Execution Pipeline
# ==============================================================================

N_SAMPLES = 2000
N_REF = 10000  
T_END = 1.0

print(f"--- Running Stiff 16D Spatial-Factorized Simulation on CPU ---")
y_noisy_ref = np.random.randn(N_REF, D_DIM) * 2.0 
y_noisy = y_noisy_ref[:N_SAMPLES]

print("1. Generating Continuous-Time Ground Truth (Ultra-fine EM, dt=0.002)...")
y_ref, _ = euler_maruyama(y_noisy_ref, dt=0.002, T=T_END)

print("\n2. Running Coarse Baseline EM (dt=0.05)...")
y_em, em_nfe = euler_maruyama(y_noisy, dt=0.05, T=T_END)

print("\n3. Running Exact SEGS Sampler (dtau=0.25)...")
y_segs, segs_nfe = segs_16d_sampler(y_noisy, dtau=0.25, T=T_END)

mmd_em = compute_mmd(y_ref[:N_SAMPLES], y_em)
mmd_segs = compute_mmd(y_ref[:N_SAMPLES], y_segs)

w2_em = compute_bures_wasserstein_distance(y_ref[:N_SAMPLES], y_em)
w2_segs = compute_bures_wasserstein_distance(y_ref[:N_SAMPLES], y_segs)

--- Running Stiff 16D Spatial-Factorized Simulation on CPU ---
1. Generating Continuous-Time Ground Truth (Ultra-fine EM, dt=0.002)...

2. Running Coarse Baseline EM (dt=0.05)...

3. Running Exact SEGS Sampler (dtau=0.25)...


In [33]:
print("\n[+] Evaluation Metrics (D=16):")
print(f"| Metric                             | EM Baseline (dt=0.05) | SEGS Exact (dtau=0.25) |")
print(f"|------------------------------------|-----------------------|------------------------|")
print(f"| Bures-Wasserstein W2 (FPD)         | {w2_em:<21.5f} | {w2_segs:<22.5f} |")
print(f"| Max Mean Discrepancy (KS Proxy)    | {mmd_em:<21.6f} | {mmd_segs:<22.6f} |")
print(f"| Avg Vector NFE per Path            | {em_nfe:<21.1f} | {segs_nfe:<22.1f} |")


[+] Evaluation Metrics (D=16):
| Metric                             | EM Baseline (dt=0.05) | SEGS Exact (dtau=0.25) |
|------------------------------------|-----------------------|------------------------|
| Bures-Wasserstein W2 (FPD)         | 0.39401               | 0.10284                |
| Max Mean Discrepancy (KS Proxy)    | 0.001150              | 0.000858               |
| Avg Vector NFE per Path            | 20.0                  | 103.3                  |
